In [36]:
import pandas as pd
import rioxarray

from scripts.dataprep import coarsen_raster, build_station_predictors, load_era5land


In [34]:
import rasterio

def grid_extent_3035(raster_paths):
    """Intersection of the bounding boxes of the given rasters (all EPSG:3035)."""
    lefts, rights, bottoms, tops = [], [], [], []
    for p in raster_paths:
        with rasterio.open(p) as src:
            b = src.bounds
            lefts.append(b.left); rights.append(b.right)
            bottoms.append(b.bottom); tops.append(b.top)
    return max(lefts), min(bottoms), min(rights), max(tops)  # intersection

city = "bern"
country = "switzerland"

"../data/imp/bern_10.tif"
base = "../data/"
raster_paths = [base+f"imp/{city}_10.tif",
                base+f"tcd/{city}_10.tif",
                base+f"dtm/{city}_30.tif",
                base+f"lcz/{city}_100.tif",]

left, bot, right, top = grid_extent_3035(raster_paths)



In [16]:
print(left, bot, right, top)

4118230.0 2600650.0 4135580.0 2690470.0


In [17]:
raster_paths = [base+f"imp/{city}_10.tif",
                base+f"tcd/{city}_10.tif",
                base+f"dtm/{city}_30.tif",
                base+f"bh/{city}_10.tif",]
import xarray as xr

rasters = []



for types, paths in zip(["imp", "tcd", 
# "dtm", 
"bh"
], raster_paths):
    rds = rioxarray.open_rasterio(paths).rio.clip_box(
        minx=left, miny=bot,
        maxx=right, maxy=top
    )
    rds = rds.squeeze(drop=True)
    rds.name = types

    if types in ["imp", "tcd"]:
        rds = rds.where(rds != 255, 0)
    elif types == "bh":
        rds = rds.where(rds != 65535, 0)

    rasters.append(rds)

merged = xr.merge(rasters)

df = merged.to_dataframe().reset_index()
print(df)

                 x          y  spatial_ref  imp  tcd           bh
0        4118235.0  2600665.0            0  NaN  NaN  1729.300049
1        4118235.0  2600695.0            0  NaN  NaN  1738.800049
2        4118235.0  2600725.0            0  NaN  NaN  1747.699951
3        4118235.0  2600755.0            0  NaN  NaN  1756.199951
4        4118235.0  2600785.0            0  NaN  NaN  1764.699951
...            ...        ...          ...  ...  ...          ...
7200245  4135575.0  2690335.0            0  NaN  NaN  1069.800049
7200246  4135575.0  2690365.0            0  NaN  NaN  1077.000000
7200247  4135575.0  2690395.0            0  NaN  NaN  1080.199951
7200248  4135575.0  2690425.0            0  NaN  NaN  1077.800049
7200249  4135575.0  2690455.0            0  NaN  NaN  1070.699951

[7200250 rows x 6 columns]


In [18]:
import os
import tempfile
import numpy as np
import pandas as pd
import rioxarray
from rasterio.enums import Resampling


def prepare_rasters(raster_paths, types, target_res, output_dir):
    """
    Coarsen rasters to target_res where possible.

    If target_res is finer than a raster's native resolution,
    the raster is left unchanged.

    Returns paths to the rasters that should be used downstream.
    """

    os.makedirs(output_dir, exist_ok=True)

    prepared_paths = []

    for typ, src_path in zip(types, raster_paths):

        with rasterio.open(src_path) as src:
            cur_x, cur_y = src.res
            cur_res = max(cur_x, cur_y)

        # Only coarsen if target is coarser than the source
        if target_res > cur_res:
            dst_path = os.path.join(
                output_dir,
                f"{city}_{typ}_{target_res}m.tif"
            )

            print(
                f"{typ}: {cur_res} m -> {target_res} m "
                f"(coarsening)"
            )

            coarsen_raster(
                src_path,
                dst_path,
                target_res
            )

            prepared_paths.append(dst_path)

        else:
            print(
                f"{typ}: {cur_res} m -> keeping native resolution "
                f"({cur_res} m)"
            )

            prepared_paths.append(src_path)

    return prepared_paths

In [19]:
DESIRED_RES = 100

In [20]:
types = ["imp", "tcd", "dtm", "bh"]

prepared_paths = prepare_rasters(
    raster_paths=raster_paths,
    types=types,
    target_res=DESIRED_RES,
    output_dir="../data/coarsened"
)

imp: 10.0 m -> 100 m (coarsening)
current pixel size is 10.0, 10.0
tcd: 10.0 m -> 100 m (coarsening)
current pixel size is 10.0, 10.0
dtm: 30.0 m -> 100 m (coarsening)
current pixel size is 30.0, 30.0
bh: 10.0 m -> 100 m (coarsening)
current pixel size is 10.0, 10.0


In [21]:
if DESIRED_RES == 10:
    raster_paths = [base+f"imp/{city}_10.tif",
                base+f"tcd/{city}_10.tif",
                base+f"dtm/{city}_30.tif",
                base+f"bh/{city}_10.tif",]
                
elif DESIRED_RES in [50, 100]:
    raster_paths = [base+f"coarsened/{city}_imp_{DESIRED_RES}m.tif",
                base+f"coarsened/{city}_tcd_{DESIRED_RES}m.tif",
                base+f"coarsened/{city}_dtm_{DESIRED_RES}m.tif",
                base+f"coarsened/{city}_bh_{DESIRED_RES}m.tif"]
import rioxarray
import xarray as xr

types = ["imp", "tcd", "dtm", "bh"]

# --- Load imp as the reference grid ---
imp = (
    rioxarray.open_rasterio(raster_paths[0])
    .rio.clip_box(minx=left, miny=bot, maxx=right, maxy=top)
    .squeeze(drop=True)
)

imp.name = "imp"

# 255 -> 0
imp = imp.where(imp != 255, 0)

rasters = [imp]

# # --- Load the other rasters ---
# for typ, raster_path in zip(types[1:], raster_paths[1:]):

#     rds = (
#         rioxarray.open_rasterio(raster_path)
#         .rio.clip_box(minx=left, miny=bot, maxx=right, maxy=top)
#         .squeeze(drop=True)
#     )

#     # Check what we're dealing with
#     print(
#         typ,
#         "CRS:", rds.rio.crs,
#         "resolution:", rds.rio.resolution(),
#         "shape:", rds.shape
#     )

#     # Put raster onto EXACTLY the imp grid
#     rds = rds.rio.reproject_match(
#         imp,
#         resampling=rioxarray.raster_array.Resampling.nearest
#     )

#     # Replace special values
#     if typ in ["tcd"]:
#         rds = rds.where(rds != 255, 0)


#     elif typ == "bh":
#         rds = rds.where(rds != 65535, 0)

#     rds.name = typ
#     rasters.append(rds)

# Merge now that everything has identical x/y grids
merged = xr.merge(rasters)

df = merged.to_dataframe().reset_index()

print(df)

                  x             y  spatial_ref  imp
0      4.118280e+06  2.660290e+06            0    0
1      4.118280e+06  2.660190e+06            0    0
2      4.118280e+06  2.660091e+06            0    0
3      4.118280e+06  2.659991e+06            0    0
4      4.118280e+06  2.659891e+06            0    0
...             ...           ...          ...  ...
30271  4.135530e+06  2.643439e+06            0   37
30272  4.135530e+06  2.643339e+06            0    0
30273  4.135530e+06  2.643239e+06            0   16
30274  4.135530e+06  2.643140e+06            0   42
30275  4.135530e+06  2.643040e+06            0   35

[30276 rows x 4 columns]


In [22]:
from pyproj import Transformer

# x/y are currently EPSG:3035
to_wgs84 = Transformer.from_crs(
    "EPSG:3035",
    "EPSG:4326",
    always_xy=True
)

# Keep original coordinates
df = df.rename(columns={
    "x": "x_orig",
    "y": "y_orig"
})

# Convert to longitude / latitude
df["longitude"], df["latitude"] = to_wgs84.transform(
    df["x_orig"].values,
    df["y_orig"].values
)

# Create station IDs
df.insert(0, "station_id", np.arange(len(df)))

# Remove columns you don't want
df = df[
    ["station_id", "x_orig", "y_orig", "latitude", "longitude"]
]

print(df)

       station_id        x_orig        y_orig   latitude  longitude
0               0  4.118280e+06  2.660290e+06  47.023350   7.335038
1               1  4.118280e+06  2.660190e+06  47.022453   7.335083
2               2  4.118280e+06  2.660091e+06  47.021556   7.335129
3               3  4.118280e+06  2.659991e+06  47.020659   7.335174
4               4  4.118280e+06  2.659891e+06  47.019762   7.335220
...           ...           ...           ...        ...        ...
30271       30271  4.135530e+06  2.643439e+06  46.877071   7.568675
30272       30272  4.135530e+06  2.643339e+06  46.876174   7.568717
30273       30273  4.135530e+06  2.643239e+06  46.875277   7.568758
30274       30274  4.135530e+06  2.643140e+06  46.874379   7.568799
30275       30275  4.135530e+06  2.643040e+06  46.873482   7.568841

[30276 rows x 5 columns]


In [24]:
if DESIRED_RES == 10:
    raster_paths = {"imp": base+f"imp/{city}_10.tif",
                    "tcd": base+f"tcd/{city}_10.tif",
                    "bh": base+f"bh/{city}_10.tif",}
                
elif DESIRED_RES in [50, 100]:
    raster_paths = {"imp": base+f"coarsened/{city}_imp_{DESIRED_RES}m.tif",
                "tcd": base+f"coarsened/{city}_tcd_{DESIRED_RES}m.tif",
                "bh": base+f"coarsened/{city}_bh_{DESIRED_RES}m.tif"}

station_predictors = build_station_predictors(df, raster_paths, radii = tuple(r for r in (50, 100, 250, 500, 750, 1000) if r > DESIRED_RES))

# merge on station_id copies all of the geospatial data from station_id to all instances of station_id in the original dataframe
df = df.merge(station_predictors.drop(columns=["latitude", "longitude", "x", "y"]),
              on="station_id", how="left")

a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station in bh is out of bounds
a station 

In [25]:
print(df)

       station_id        x_orig        y_orig   latitude  longitude  \
0               0  4.118280e+06  2.660290e+06  47.023350   7.335038   
1               1  4.118280e+06  2.660190e+06  47.022453   7.335083   
2               2  4.118280e+06  2.660091e+06  47.021556   7.335129   
3               3  4.118280e+06  2.659991e+06  47.020659   7.335174   
4               4  4.118280e+06  2.659891e+06  47.019762   7.335220   
...           ...           ...           ...        ...        ...   
30271       30271  4.135530e+06  2.643439e+06  46.877071   7.568675   
30272       30272  4.135530e+06  2.643339e+06  46.876174   7.568717   
30273       30273  4.135530e+06  2.643239e+06  46.875277   7.568758   
30274       30274  4.135530e+06  2.643140e+06  46.874379   7.568799   
30275       30275  4.135530e+06  2.643040e+06  46.873482   7.568841   

       imp_nearest  imp_buf250  imp_buf500  imp_buf750  imp_buf1000  \
0              0.0    0.000000    0.000000    0.000000     0.055556   
1    

In [27]:
single_paths = {
    "dtm":  f"../data/dtm/{city}_30.tif",
    "lcz": f"../data/lcz/{city}_100.tif",
}
single_predictors = build_station_predictors(df, single_paths, radii=())

df = df.merge(single_predictors.drop(columns=["latitude", "longitude", "x", "y"]),
              on="station_id", how="left")

In [28]:
df = df.dropna()

In [32]:
# pick one station's first row (any timestamp works — geo predictors are time-invariant)
stations = df["station_id"].unique()
sid = stations[1]        # change this index: 0, 1, 2, ... = different stations
row = df[df["station_id"] == sid].iloc[0]

# the columns we added (everything from the rasters)
geo_cols = [c for c in df.columns
            if c not in ("station_id",
                        )]

print(f"Station: {sid}")
for c in geo_cols:
    print(f"  {c:20s} {row[c]}")

Station: 1
  x_orig               4118279.856321839
  y_orig               2660190.4310344825
  latitude             47.02245305713248
  longitude            7.335083211788628
  imp_nearest          0.0
  imp_buf250           0.0
  imp_buf500           0.0
  imp_buf750           0.0
  imp_buf1000          0.07000000029802322
  tcd_nearest          99.0
  tcd_buf250           98.0
  tcd_buf500           94.90322875976562
  tcd_buf750           93.0
  tcd_buf1000          83.45999908447266
  bh_nearest           0.0
  bh_buf250            0.0
  bh_buf500            0.0
  bh_buf750            0.0
  bh_buf1000           0.0
  dtm_nearest          760.2000122070312
  lcz_nearest          11.0


In [35]:
# city center coords for ERA5-Land nearest-cell selection
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="get_cc_coords")

location = geolocator.geocode(city + "," + country)

lat0, lon0 = location.latitude, location.longitude

print(lat0, lon0)



46.9484742 7.4521749


In [39]:
# obtain what days our data is measured over
needed = ['2024-07-25']
met, era5_elev = load_era5land(f"../../raw_data/era5land/",
                               f"../../raw_data/era5land/geopotential.nc",
                               lat0, lon0, needed_dates=needed)




ERA5-Land (1 days): 100%|██████████| 1/1 [00:23<00:00, 23.26s/it]


In [40]:
met = deaccumulate(met, "ssrd")                                                  # 3
met = deaccumulate(met, "tp")
met = transform_met(met)                                                         # 4



NameError: name 'deaccumulate' is not defined